# Is `readable ≠ grabbable` a property of the *rendering*, before any learning?

**Direction:** `research/directions/orthogonal-edits.md` · `[reframe]` · sub-Q 3.
**Branch:** `orthogonal_edit_analysis`. Origin: Sevan, 2026-08-05.

## The question

`transformers/transformer_world_state.ipynb` §6 measured, **inside a trained network**, the angle between
(i) the directions a linear position probe can write in — its **row space** — and (ii) the direction the
decoder says would actually change the render toward the target. It came out **87–90°**, and the row-space
fraction of the decoder's preferred direction was **at or below chance**. That is why pseudoinverse
injection is inert.

Two explanations were left open:

| | claim | consequence if true |
|---|---|---|
| **inherited** | the misalignment is baked into the rendering geometry | any model trained on this world has it; no architecture or probe escapes it |
| **chosen** | the network happened to pick an internal layout where these directions disagree | a fact about these models, possibly fixable by training |

**This notebook removes the network entirely** and asks the same question of raw observations. If the angle
is still ~90° with no learning anywhere, the misalignment is inherited.

## Why one would expect it — the hand calculation

An object covers `n ≈ 21` rays at intensity `0.4`. Call the observation `A` (a flat plateau) and call the
change produced by nudging the object one ray to the right `B`. `B` is `−0.4` on the ray that goes dark,
`+0.4` on the ray that lights up, and **exactly zero on the 20 rays in between**. So

$$A\cdot B = (0.4)(-0.4) = -0.16,\qquad \|A\| = 0.4\sqrt{21},\qquad \|B\| = 0.4\sqrt{2}$$
$$\cos = \frac{-0.16}{1.83 \times 0.57} = -0.154 \;\;\Rightarrow\;\; 99°$$

Generally `cos ≈ −√(k / 2n)` for an object covering `n` rays and shifting by `k`. **A wide plateau is nearly
perpendicular to the thin spikes at its own edges.** Position information lives across the whole plateau;
moving the object requires changing only the edges.

## What is measured here

Everything is the exact structural analogue of §6, with `h ∈ R^256` replaced by the raw observation
`o ∈ R^128` and the world model replaced by nothing at all.

| §6 (inside the network) | here (no network) |
|---|---|
| probe `h → position` | probe `clean_obs → position` |
| `Δh_pinv = (target − (A h + b)) A⁺` | `Δo_pinv = (target − (A o + b)) A⁺` |
| decoder's descent direction in `h` | the **required** change `Δo_true = gt_edited − gt_unedited` |
| row-space chance `√(4/256) = 0.125` | row-space chance `√(4/128) = 0.177` |

## Definitions

| term | formula | units | notes |
|---|---|---|---|
| **required change** `Δo_true` | `gt_edited − gt_unedited`, both **clean** renders | direction in `R^128` | exactly what any editor must produce at the edit frame: the observation-space vector from the world where the teleport did not happen to the world where it did |
| **pseudoinverse direction** `Δo_pinv` | `(tgt_pos − (A o + b)) A⁺` | direction in `R^128` | what readout injection applies. Lies in `row(A)` **by construction** |
| **cosine** | `⟨u,v⟩ / (‖u‖‖v‖)`, **per sample then averaged** | −1…+1 | never on averaged vectors. Reported with the **angle**, since cos 0.9 is 26°, not "90% similar" |
| **row-space fraction** | `‖Qᵀ Δo_true‖ / ‖Δo_true‖`, `Q` = orthonormal basis of `row(A)` | 0…1 | the share of the required change that injection can reach **at all** — the hard ceiling |
| **chance level** | `√(d/R)`, `d = 4` probe outputs, `R = 128` rays | 0…1 | a random direction already has this much of its norm in any 4-d subspace: **0.177**. Always report the ratio to chance |
| **shuffled control** | the same cosine with `Δo_pinv` from a *different* sample | −1…+1 | the empirical null; its mean is **0**, not `1/√R` |
| **nudge** | one object moved `0.05` sim units (≈ 1 ray) | — | the pedagogical case from the hand calculation above |
| **teleport** | the actual `edits`-split intervention at frame 20 | — | the case that matches §6 |

**Data.** `datasets/4_fixed_refl_inview`, `edits` split, edit frame 20, 2 objects, `obs_res = 128`,
fixed reflectivities (so the clean render is a function of positions alone). **N = 2000** samples — this is
cheap because nothing is trained. Probe fit on the `test` split, applied to `edits`; never fit and evaluated
on the same rows.

In [ ]:
# [1] Setup: dataset, renderer, and the linear position probe fit on RAW CLEAN OBSERVATIONS.
import sys, json
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
import numpy as np, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

from pim.world_models import load_dataset
from pim.simulator.sim import SimConfig
from pim.simulator.renderer import render_frame
from pim.figures.theme import style_ax

np.random.seed(0)
N_OBJ, N = 2, 2000
OUT = "/tmp/orthogonal_edits"
import os; os.makedirs(OUT, exist_ok=True)

bundle = load_dataset("../../../../datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
edits, test = bundle.edits, bundle.test
ef = edits.edit_frame; sim = test.config["dataset"]["sim"]; R = edits.obs_res
CFG = SimConfig(**{k: v for k, v in sim.items() if k in SimConfig.__dataclass_fields__})
CFG.n_objects = N_OBJ
RADII = np.full(N_OBJ, sim["radius"])
REFL = np.linspace(sim["refl_min"], sim["refl_max"], N_OBJ)   # fixed_reflectivities=True

def render(pos):
    """Clean render of one frame: (n_obj, 2) positions -> (R,) intensity. No noise."""
    c = SimConfig(**{**CFG.__dict__, "obs_noise_std": 0.0})
    return render_frame(np.asarray(pos, float), RADII, REFL, c)[2]

def render_many(P):
    return np.stack([render(p) for p in P]).astype(np.float32)

# ── the probe: clean observation -> the 4 position coordinates, fit on the TEST split ──
tr_obs = test.clean_obs[:4000, ef, :].astype(np.float32)
tr_pos = test.positions[:4000, ef, :N_OBJ, :].astype(np.float32).reshape(-1, N_OBJ*2)
Aug = np.concatenate([tr_obs, np.ones((len(tr_obs), 1), np.float32)], 1)
sol, *_ = np.linalg.lstsq(Aug, tr_pos, rcond=None)
Wp_, b_ = sol[:-1], sol[-1]                       # W: (R, 4)
Wpinv = np.linalg.pinv(Wp_)                       # (4, R)
Q, _ = np.linalg.qr(Wp_)                          # (R, 4) orthonormal basis of row(A)
CHANCE = np.sqrt(Wp_.shape[1] / R)

he_obs = edits.clean_obs[:N, ef, :].astype(np.float32)
he_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32).reshape(-1, N_OBJ*2)
r2 = 1 - ((he_obs @ Wp_ + b_ - he_pos)**2).sum() / ((he_pos - he_pos.mean(0))**2).sum()
print(f"probe: clean_obs (R={R}) -> position ({N_OBJ*2} dims), fit on test, evaluated on edits")
print(f"  held-out R^2 = {r2:.3f}   (only WEAKLY linearly readable -- see the control below)")
print(f"  row(A) is {Wp_.shape[1]}-dimensional inside R^{R};  chance = sqrt(d/R) = {CHANCE:.3f}")

# Control: is that R^2 low because the probe is under-fit, or because the map is nonlinear?
# A nonlinear (MLP) probe on the SAME raw observations answers it.
from eval_controls import _fit_mlp, _apply, _r2      # the repo's canonical MLP probe
# Standardise the TARGETS before fitting. `_fit_mlp` is tuned for h-vectors; the position targets
# here have scale ~8 (the y coordinate) and without this the net does not converge at all
# (measured R^2 = -0.58, worse than predicting the mean -- a failed fit, not a result).
mu_y, sd_y = tr_pos.mean(0), tr_pos.std(0) + 1e-8
mlp = _fit_mlp(tr_obs, ((tr_pos - mu_y) / sd_y).astype(np.float32))
r2_mlp = _r2(_apply(mlp, he_obs) * sd_y + mu_y, he_pos)
names4 = ["obj0 x", "obj0 y", "obj1 x", "obj1 y"]
per = 1 - ((he_obs @ Wp_ + b_ - he_pos)**2).sum(0) / ((he_pos - he_pos.mean(0))**2).sum(0)
print(f"\n  CONTROL - is the low R^2 under-fitting, or is the map genuinely nonlinear?")
print(f"    linear probe   R^2 = {r2:.3f}     per coordinate: "
      + "  ".join(f"{n} {v:.2f}" for n, v in zip(names4, per)))
print(f"    MLP probe      R^2 = {r2_mlp:.3f}     (same inputs, same fit/eval split)")
print(f"    => a nonlinear readout is far stronger, so the linear probe's weakness is LINEARITY, not")
print(f"       under-fitting. Note obj0 (reflectivity {REFL[0]}) is much less linearly readable than obj1")
print(f"       ({REFL[1]}): a linear map keys on brightness to tell the two plateaus apart.")
print(f"       Readout injection REQUIRES a linear probe -- that is what makes the pseudoinverse defined --")
print(f"       so the linear row space is the right object to test. And the conclusion does NOT depend on")
print(f"       probe quality: injection hits its own readout target exactly and still moves nothing.")

---
## §1 — The picture, on one sample

Panel (a) is the whole argument in one plot: the observation is a **wide plateau**, and the change that moves
the object is **two thin spikes at its edges**, with nothing in between.

In [ ]:
# [2] Fig 1 — one sample: the observation, the nudge direction, and the required teleport direction.
i = 0
oe = edits.edit_object[:N].astype(int)
pre_pos = edits.positions[:N, ef-1, :N_OBJ, :].astype(np.float32)
tgt_pos = edits.positions[:N, ef,   :N_OBJ, :].astype(np.float32)
vel = None
with h5py.File(edits.h5_path, "r") as f:
    VEL = f["velocities"][:N, ef, :N_OBJ, :].astype(np.float32)

# unedited world = the teleport never happened: the edited object continues on its own velocity
un_pos = tgt_pos.copy()
for k in range(N):
    un_pos[k, oe[k]] = pre_pos[k, oe[k]] + VEL[k, oe[k]]
GT_ED = render_many(tgt_pos)          # the world where the edit happened
GT_UN = render_many(un_pos)           # the world where it did not
D_TRUE = GT_ED - GT_UN                # the REQUIRED change in observation space

nud_pos = un_pos.copy()
nud_pos[:, :, 0] += 0.05              # ~1 ray nudge of BOTH objects in x
D_NUDGE = render_many(nud_pos) - GT_UN

plt.style.use("default")
fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.3))
rays = np.arange(R)
ax[0].plot(rays, GT_UN[i], color="#0072B2", lw=1.8, label="observation  (where the object is)")
ax[0].plot(rays, D_NUDGE[i], color="#D55E00", lw=1.8,
           label="change from a 1-ray nudge  (how to move it)")
ax[0].axhline(0, color="0.6", lw=0.8)
ax[0].set_xlabel("ray"); ax[0].set_ylabel("intensity")
ax[0].set_title("(a) the plateau and its edges, one sample", fontsize=9.5)
ax[0].legend(fontsize=8)
cn = (GT_UN * D_NUDGE).sum(1) / (np.linalg.norm(GT_UN, axis=1)*np.linalg.norm(D_NUDGE, axis=1) + 1e-12)
cn = cn[np.linalg.norm(D_NUDGE, axis=1) > 1e-9]
ax[1].hist(cn, bins=40, color="#0072B2", alpha=0.85)
ax[1].axvline(cn.mean(), color="#D55E00", lw=2)
ax[1].annotate(f"mean {cn.mean():+.3f}   ({np.degrees(np.arccos(np.clip(cn.mean(),-1,1))):.0f}°)",
               xy=(cn.mean(), 1.02), xycoords=("data", "axes fraction"), fontsize=9,
               color="#D55E00", ha="center", va="bottom")
ax[1].axvline(0, color="0.5", ls=":", lw=1.2)
ax[1].set_xlabel("cosine(observation, nudge direction)"); ax[1].set_ylabel("samples")
ax[1].set_title(f"(b) over {len(cn)} samples — no network involved", fontsize=9.5)
for a_ in ax: a_.grid(alpha=0.3); style_ax(a_)
fig.suptitle("Fig 1 — an object's own image is nearly perpendicular to the change that moves it",
             y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_plateau_vs_edges.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
keep_n = np.linalg.norm(D_NUDGE, axis=1) > 1e-9
n_cov = (GT_UN > 1e-6).sum(1)[keep_n]                    # rays covered, PER SAMPLE
pred = -np.sqrt(1.0 / (2.0 * np.maximum(n_cov, 1)))      # the hand formula, per sample
print(f"hand formula cos ~ -sqrt(k/(2n)) with k=1, evaluated PER SAMPLE on its own ray count n:")
print(f"  rays covered n: mean {n_cov.mean():.1f} (range {n_cov.min()}-{n_cov.max()})")
print(f"  predicted cosine {pred.mean():+.3f}   measured cosine {cn.mean():+.3f}")

---
## §2 — The network-free version of §6

Same construction as `transformer_world_state.ipynb` §6, with the network deleted: does the direction an
editor *must* produce lie in the subspace a linear position probe can write in?

In [ ]:
# [3] Fig 2 — row-space geometry of the required change, with no model anywhere.
def cos_rows(U, V):
    u = U / (np.linalg.norm(U, axis=1, keepdims=True) + 1e-12)
    v = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)
    return (u * v).sum(1)

tgt4 = tgt_pos.reshape(N, N_OBJ*2)
D_PINV = (tgt4 - (GT_UN @ Wp_ + b_)) @ Wpinv          # what readout injection would apply, in R^128
rs = np.random.default_rng(0).permutation(N)

RES = {}
for name, Dv in [("teleport (matches §6)", D_TRUE), ("1-ray nudge", D_NUDGE)]:
    keep = np.linalg.norm(Dv, axis=1) > 1e-9
    Dk, Pk = Dv[keep], D_PINV[keep]
    RES[name] = dict(
        cos=cos_rows(Dk, Pk),
        shuf=cos_rows(Dk, D_PINV[rs][:len(Dk)]),
        frac=np.linalg.norm(Dk @ Q, axis=1) / np.linalg.norm(Dk, axis=1))

plt.style.use("default")
fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.4))
xs = np.arange(len(RES)); w = 0.35
names = list(RES)
ax[0].bar(xs-w/2, [RES[k]["cos"].mean() for k in names], w,
          yerr=[RES[k]["cos"].std() for k in names], capsize=4,
          color="#0072B2", label="cosine(required change, pseudoinverse direction)")
ax[0].bar(xs+w/2, [RES[k]["shuf"].mean() for k in names], w,
          yerr=[RES[k]["shuf"].std() for k in names], capsize=4,
          color="0.65", label="shuffled control (empirical null)")
ax[0].axhline(0, color="0.4", lw=1.0)
ax[0].set_xticks(xs); ax[0].set_xticklabels(names, fontsize=9)
ax[0].set_ylim(-1.05, 1.05); ax[0].set_ylabel("cosine")
ax[0].set_title("(a) does injection push the way the render needs?", fontsize=9.5)
ax[0].legend(fontsize=7.5, loc="upper right")
sec = ax[0].secondary_yaxis("right", functions=(lambda c: np.degrees(np.arccos(np.clip(c, -1, 1))),
                                                lambda a: np.cos(np.radians(a))))
sec.set_ylabel("angle (degrees)", fontsize=9)
ax[1].bar(xs, [RES[k]["frac"].mean() for k in names], 0.5,
          yerr=[RES[k]["frac"].std() for k in names], capsize=4, color="#009E73")
ax[1].axhline(CHANCE, color="#D55E00", ls=":", lw=1.8)
ax[1].annotate(f"chance for a random direction: √(4/128) = {CHANCE:.3f}",
               xy=(0.03, CHANCE), xycoords=("axes fraction", "data"), fontsize=8,
               color="#D55E00", va="bottom")
ax[1].set_xticks(xs); ax[1].set_xticklabels(names, fontsize=9)
ax[1].set_ylabel("fraction of the required change inside row(A)")
ax[1].set_title("(b) how much of it can injection reach at all?", fontsize=9.5)
for a_ in ax: a_.grid(alpha=0.3, axis="y"); style_ax(a_)
fig.suptitle(f"Fig 2 — the same geometry as §6, measured on raw observations with no model of any kind "
             f"(N = {N}, per-sample then averaged)", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_rowspace_no_model.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

rows = ["| intervention | cosine(required, pseudoinverse) | angle | shuffled control | row-space fraction "
        "| ÷ chance |", "|---|---|---|---|---|---|"]
for k in names:
    d = RES[k]
    rows.append(f"| {k} | {d['cos'].mean():+.3f} ± {d['cos'].std():.3f} "
                f"| {np.degrees(np.arccos(np.clip(d['cos'].mean(), -1, 1))):.1f}° "
                f"| {d['shuf'].mean():+.3f} ± {d['shuf'].std():.3f} "
                f"| {d['frac'].mean():.3f} ± {d['frac'].std():.3f} | {d['frac'].mean()/CHANCE:.2f}× |")
rows.append("| *(reference)* `transformer W16`, last residual point, **inside the network** "
            "| +0.014 ± 0.042 | 89.2° | +0.001 ± 0.040 | 0.071 ± 0.035 | 0.57× |")
rows.append("| *(reference)* `GRU H=256`, **inside the network** "
            "| +0.034 ± 0.100 | 88.1° | −0.017 ± 0.098 | 0.189 ± 0.074 | 1.51× |")
display(Markdown("**Table 1 — the network-free measurement, against the in-network §6 numbers.** The two "
                 "reference rows are copied from `transformers/transformer_world_state.ipynb` Table 6 and "
                 "are measured in `h`-space (`R = 256`, chance 0.125); the rows above them are measured in "
                 "observation space (`R = 128`, chance 0.177). The *fractions* are therefore not directly "
                 "comparable across that line — the **÷ chance** column is.\n\n" + "\n".join(rows)))

---
## §3 — What injection actually renders

The most direct check available, and one the in-network experiment cannot make: apply the pseudoinverse
edit **to the observation itself** and simply look at it. The probe then reads the target position exactly.
The question is whether the picture shows the object in a new place.

In [ ]:
# [4] Fig 3 — apply readout injection directly to the observation and look at the result.
O_INJ = GT_UN + D_PINV
read_before = GT_UN @ Wp_ + b_
read_after  = O_INJ @ Wp_ + b_
print(f"probe readout error to target:  before {np.linalg.norm(read_before-tgt4,axis=1).mean():.3f}"
      f"  ->  after {np.linalg.norm(read_after-tgt4,axis=1).mean():.2e} sim units  (the write lands exactly)")

SH = [0, 1, 2]
plt.style.use("default")
fig, ax = plt.subplots(1, len(SH), figsize=(5.2*len(SH), 3.9), squeeze=False)
ax = ax[0]
for j, smp in enumerate(SH):
    o_ = oe[smp]
    ax[j].plot(rays, GT_UN[smp], color="0.55", lw=1.6, label="before (unedited world)")
    ax[j].plot(rays, GT_ED[smp], color="#009E73", lw=2.2, label="target (edited world)")
    ax[j].plot(rays, O_INJ[smp], color="#CC79A7", lw=1.8, ls="--",
               label="after pseudoinverse injection")
    ax[j].set_xlabel("ray"); ax[j].set_ylabel("intensity" if j == 0 else "")
    ax[j].set_title(f"sample {smp} · teleport "
                    f"{np.linalg.norm(tgt_pos[smp,o_]-un_pos[smp,o_]):.1f} units", fontsize=9.5)
    ax[j].grid(alpha=0.3); style_ax(ax[j])
h_, l_ = ax[0].get_legend_handles_labels()
fig.legend(h_, l_, loc="upper center", ncol=3, fontsize=9, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle("Fig 3 — pseudoinverse injection applied directly to the observation: the probe now reads the "
             "target exactly, and the picture does not move", y=1.13, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.90])
fig.savefig(f"{OUT}/fig3_injected_observation.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

def rmse(a, b): return float(np.sqrt(((a-b)**2).mean()))
print(f"\nRMSE to the target (edited) render, over N = {N}:")
print(f"  doing nothing                 {rmse(GT_UN, GT_ED):.4f}")
print(f"  after pseudoinverse injection {rmse(O_INJ, GT_ED):.4f}")
print(f"  => injection closed {100*(1 - rmse(O_INJ, GT_ED)/rmse(GT_UN, GT_ED)):.1f}% of the gap "
      "to the world it was asked to produce.")

---
## §4 — Summary

In [ ]:
# [5] Computed summary.
print("========== Summary — computed here, no model of any kind ==========\n")
print(f"1. A linear position readout on the raw render is weak (R^2 = {r2:.3f}); a nonlinear one is")
print(f"   much stronger (R^2 = {r2_mlp:.3f}), so the render->position map is strongly nonlinear.")
print(f"   Readout injection nevertheless REQUIRES a linear probe, and it hits that probe's target")
print(f"   exactly, so the result below does not hinge on probe quality.")
te = RES["teleport (matches §6)"]; nu = RES["1-ray nudge"]
print(f"\n2. But the change an editor must produce is nearly perpendicular to what a readout")
print(f"   injection can apply:")
for k in names:
    d = RES[k]
    print(f"     {k:<22s} cosine {d['cos'].mean():+.3f} "
          f"({np.degrees(np.arccos(np.clip(d['cos'].mean(),-1,1))):.0f}deg), "
          f"shuffled control {d['shuf'].mean():+.3f}")
print(f"\n3. Row-space fraction of the required change (chance = {CHANCE:.3f}):")
for k in names:
    d = RES[k]
    print(f"     {k:<22s} {d['frac'].mean():.3f}  = {d['frac'].mean()/CHANCE:.2f}x chance")
print(f"\n4. Rendering the injected observation: it closes "
      f"{100*(1 - rmse(O_INJ, GT_ED)/rmse(GT_UN, GT_ED)):.1f}% of the gap to the target world,")
print(f"   despite the probe reading the target position to "
      f"{np.linalg.norm(read_after-tgt4,axis=1).mean():.0e} sim units.")
near_orth = abs(te["cos"].mean()) < 0.25
at_chance = te["frac"].mean() / CHANCE < 1.6
print("\n5. VERDICT")
if near_orth and at_chance:
    print("     INHERITED. The misalignment is present in the raw observation space, before any")
    print("     learning. A linear position probe reads the plateau; moving the object requires")
    print("     changing the edges; those are near-orthogonal. No world model, architecture or")
    print("     probe-fitting choice can escape this - it is a property of the rendering.")
elif near_orth:
    print("     PARTLY INHERITED. The directions are near-orthogonal in observation space, but the")
    print("     row-space fraction is enriched over chance - read Fig 2b before concluding.")
else:
    print("     NOT INHERITED. Raw observation space does not show the misalignment, so the networks")
    print("     introduced it. Section 6 is then a fact about the models, not about the world.")